# FINAL2 — V3 Choice-Permutation TTA

현재 최고 Public Score:

```text
V3 = 0.95918
Qwen3-VL-8B
1536 visual tokens
Full Image + Question-aware Top-1 OCR Crop
```

지금까지 확인한 결과:

- 2048 해상도: 하락
- OCR text 직접 삽입: 하락
- Top-2 OCR Crop TTA: 하락
- V1.1 + V3 ensemble: 하락
- 4-way CE: 하락
- Qwen3-VL-30B-A3B: 크게 하락

따라서 **V3의 입력 구조와 weight를 전혀 바꾸지 않고**,  
multiple-choice 모델에서 생길 수 있는 **a/b/c/d 위치 편향만 줄이는 TTA**를 시도합니다.

---

## 핵심 아이디어

원래 보기:

```text
(a) A
(b) B
(c) C
(d) D
```

추가 추론 1:

```text
(a) B
(b) C
(c) D
(d) A
```

추가 추론 2:

```text
(a) C
(b) D
(c) A
(d) B
```

추가 추론 3:

```text
(a) D
(b) A
(c) B
(d) C
```

즉 기존 V3 prediction까지 합치면 **원래 선택지 A/B/C/D가 a/b/c/d 위치에 정확히 한 번씩 등장**합니다.

모델이 `b`라는 위치를 좋아하는 편향이 있어도 4-view 평균에서는 상쇄될 수 있습니다.

---

## 시간을 줄이는 방법

모든 문제를 3번 더 돌리지 않습니다.

먼저 기존 V3의 `a/b/c/d` margin이 낮은 문제만 추가 추론합니다.

Dev에서는 margin <= 0.50 문제를 미리 계산하고,
그 안에서 실제로 사용할 threshold와 base/TTA weight를 탐색합니다.

Test에서는 Dev가 선택한 threshold에 해당하는 문제만 추가 추론합니다.

---

## 제출 파일

최종적으로:

```text
00_v3_reproduction.csv
01_perm_conservative.csv   # V3 대비 최대 3% 변경
02_perm_balanced.csv       # 최대 8% 변경
03_perm_aggressive.csv     # 최대 15% 변경
04_perm_vote_switch.csv    # 3개 permutation이 강하게 합의할 때만 변경
```

를 만듭니다.

시간이 정말 없고 제출 1번만 가능하다면 **01 또는 02**를 우선 확인합니다.

# 0. GPU

In [ ]:
!nvidia-smi

Tue Sep 22 13:48:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   34C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# 1. 설치

In [ ]:
%pip install -q \
    "transformers==4.57.0" \
    "accelerate>=1.0.0" \
    "peft>=0.13.2,<0.19" \
    "huggingface_hub>=0.25.0,<1.0" \
    "qwen_vl_utils>=0.0.14"

%pip install -q \
    "pandas==2.2.2" \
    "pillow==11.3.0" \
    tqdm psutil

%pip uninstall -y torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.1 MB/s eta 0:00:00
Reason for being yanked: Error in the setup causing installation issues
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 187.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 162.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 95.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

# 2. 기본 설정

In [ ]:
import os
import gc
import re
import json
import random
from collections import Counter
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CHOICES = ["a", "b", "c", "d"]
SCORE_COLS = ["score_a", "score_b", "score_c", "score_d"]

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

MIN_VISUAL_TOKENS = 256
MAX_VISUAL_TOKENS = 1536
MIN_PIXELS = MIN_VISUAL_TOKENS * 32 * 32
MAX_PIXELS = MAX_VISUAL_TOKENS * 32 * 32

# Dev에서 이 margin 이하만 permutation 추론
DEV_MAX_BASE_MARGIN = 0.50

# Dev search 후보
MARGIN_THRESHOLDS = [0.08, 0.12, 0.16, 0.20, 0.25, 0.30, 0.40, 0.50]
BASE_WEIGHTS = [0.25, 0.35, 0.45, 0.55, 0.65]

# V3 OCR crop 설정 그대로
OCR_MIN_CONF = 0.20
CROP_EXPAND_X = 2.3
CROP_EXPAND_Y = 3.0
MIN_CROP_WIDTH = 480
MIN_CROP_HEIGHT = 320
MIN_RELEVANCE_SCORE = 0.18

# 3 cyclic permutation
PERMUTATIONS = [
    [1, 2, 3, 0],
    [2, 3, 0, 1],
    [3, 0, 1, 2],
]

print("Model:", MODEL_ID)
print("Visual tokens:", MAX_VISUAL_TOKENS)
print("Dev max margin:", DEV_MAX_BASE_MARGIN)

Model: Qwen/Qwen3-VL-8B-Instruct
Visual tokens: 1536
Dev max margin: 0.5


# 3. Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


# 4. 경로 설정

현재 데이터 구조 기준:

```text
/content/drive/MyDrive/data/
├── dev.csv
├── test.csv
├── dev/
└── test/
```

로 설정했습니다.

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/data"

DEV_CSV = os.path.join(DATA_ROOT, "dev.csv")
TEST_CSV = os.path.join(DATA_ROOT, "test.csv")

V3_DIR = "/content/drive/MyDrive/Qwen3VL_VQA_V3_1536_OCR_CROP"
V3_BEST_ADAPTER = os.path.join(V3_DIR, "best_lora")

V3_DEV_CANDIDATES = [
    os.path.join(V3_DIR, "dev_predictions_epoch_2.csv"),
    os.path.join(V3_DIR, "dev_predictions_epoch_1.csv"),
]

V3_TEST_CANDIDATES = [
    os.path.join(V3_DIR, "test_predictions_with_logits_and_crop.csv"),
    os.path.join(V3_DIR, "test_predictions_with_logits.csv"),
    os.path.join(V3_DIR, "test_predictions.csv"),
]

OCR_CACHE_CANDIDATES = [
    "/content/drive/MyDrive/Qwen3VL_VQA_V2_2048_OCR/ocr_cache",
    os.path.join(V3_DIR, "ocr_cache"),
]

OUTPUT_DIR = "/content/drive/MyDrive/Qwen3VL_FINAL2_PERM_TTA"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def first_existing(paths):
    for p in paths:
        if os.path.isfile(p):
            return p
    return None


def first_existing_dir(paths):
    for p in paths:
        if os.path.isdir(p):
            return p
    return None


V3_DEV = first_existing(V3_DEV_CANDIDATES)
V3_TEST = first_existing(V3_TEST_CANDIDATES)
OCR_CACHE_DIR = first_existing_dir(OCR_CACHE_CANDIDATES)

print("Dev CSV :", DEV_CSV)
print("Test CSV:", TEST_CSV)
print("V3 Dev  :", V3_DEV)
print("V3 Test :", V3_TEST)
print("V3 LoRA :", V3_BEST_ADAPTER)
print("OCR     :", OCR_CACHE_DIR)

assert os.path.isfile(DEV_CSV)
assert os.path.isfile(TEST_CSV)
assert V3_DEV is not None
assert V3_TEST is not None
assert os.path.isdir(V3_BEST_ADAPTER)
assert OCR_CACHE_DIR is not None

Dev CSV : /content/drive/MyDrive/data/dev.csv
Test CSV: /content/drive/MyDrive/data/test.csv
V3 Dev  : /content/drive/MyDrive/Qwen3VL_VQA_V3_1536_OCR_CROP/dev_predictions_epoch_2.csv
V3 Test : /content/drive/MyDrive/Qwen3VL_VQA_V3_1536_OCR_CROP/test_predictions_with_logits_and_crop.csv
V3 LoRA : /content/drive/MyDrive/Qwen3VL_VQA_V3_1536_OCR_CROP/best_lora
OCR     : /content/drive/MyDrive/Qwen3VL_VQA_V2_2048_OCR/ocr_cache


# 5. 데이터 / V3 logits 로드

In [ ]:
dev_df = pd.read_csv(DEV_CSV)
test_df = pd.read_csv(TEST_CSV)

v3_dev = pd.read_csv(V3_DEV)
v3_test = pd.read_csv(V3_TEST)

for df in [dev_df, test_df, v3_dev, v3_test]:
    df["id"] = df["id"].astype(str)

v3_dev = dev_df[["id"]].merge(v3_dev, on="id", how="left")
v3_test = test_df[["id"]].merge(v3_test, on="id", how="left")

assert not v3_dev[SCORE_COLS].isna().any().any()
assert not v3_test[SCORE_COLS].isna().any().any()

print("Dev :", len(dev_df))
print("Test:", len(test_df))

Dev : 2683
Test: 6714


# 6. V3 probability / margin

In [ ]:
def logits_to_probs(df):
    x = df[SCORE_COLS].to_numpy(dtype=np.float64)
    x -= x.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / ex.sum(axis=1, keepdims=True)


def prob_features(p):
    s = np.sort(p, axis=1)[:, ::-1]
    idx = p.argmax(axis=1)

    return {
        "pred": np.array([CHOICES[i] for i in idx]),
        "confidence": s[:, 0],
        "margin": s[:, 0] - s[:, 1],
    }


p3_dev = logits_to_probs(v3_dev)
p3_test = logits_to_probs(v3_test)

f3_dev = prob_features(p3_dev)
f3_test = prob_features(p3_test)

print("Dev margin quantiles:")
print(pd.Series(f3_dev["margin"]).quantile([0,.1,.25,.5,.75,.9,1]))

Dev margin quantiles:
0.00    0.000000
0.10    0.407658
0.25    0.847137
0.50    0.991942
0.75    0.998998
0.90    0.999719
1.00    0.999976
dtype: float64


# 7. Dev 평가

In [ ]:
DEV_ANSWER_COLS = [
    c for c in ["answer1","answer2","answer3","answer4","answer5"]
    if c in dev_df.columns
]

def valid_labels(row):
    labels = []
    for c in DEV_ANSWER_COLS:
        x = row[c]
        if pd.notna(x):
            x = str(x).strip().lower()
            if x in CHOICES:
                labels.append(x)
    return labels


def evaluate_dev(preds):
    consensus = []
    majority = []
    high_conf = []

    for i, (_, row) in enumerate(dev_df.iterrows()):
        labels = valid_labels(row)
        if not labels:
            continue

        pred = preds[i]
        consensus.append(sum(x == pred for x in labels) / len(labels))

        cnt = Counter(labels)
        mx = max(cnt.values())
        winners = [k for k,v in cnt.items() if v == mx]

        if len(winners) == 1:
            hit = int(pred == winners[0])
            majority.append(hit)

            if mx / len(labels) >= .60:
                high_conf.append(hit)

    return {
        "consensus": float(np.mean(consensus)),
        "majority": float(np.mean(majority)),
        "high_conf": float(np.mean(high_conf)),
    }


print("V3 Dev:", evaluate_dev(f3_dev["pred"]))

V3 Dev: {'consensus': 0.4760101952007957, 'majority': 0.6872743682310469, 'high_conf': 0.7006869479882237}


# 8. V3 OCR Top-1 Crop 복원

In [ ]:
def resolve_image_path(path):
    path = str(path)
    return path if os.path.isabs(path) else os.path.join(DATA_ROOT, path)


OCR_JSON_MAP = {}

for split in ["dev", "test", "train"]:
    p = os.path.join(OCR_CACHE_DIR, f"ocr_{split}.csv")

    if not os.path.isfile(p):
        continue

    cache = pd.read_csv(p)

    for _, r in cache.iterrows():
        try:
            items = json.loads(str(r.get("ocr_json", "[]")))
        except Exception:
            items = []

        OCR_JSON_MAP[str(r["id"])] = items

print("OCR JSON map:", len(OCR_JSON_MAP))


def normalize_for_match(text):
    text = str(text).lower()
    text = re.sub(r"\s+", "", text)
    return re.sub(r"[^\w가-힣0-9]", "", text)


def extract_digits(text):
    return re.sub(r"\D", "", str(text))


def seq_sim(a, b):
    a = normalize_for_match(a)
    b = normalize_for_match(b)
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def ocr_relevance_score(ocr_text, conf, row):
    question = str(row["question"])
    choices = [str(row[c]) for c in ["a","b","c","d"]]

    q_score = seq_sim(ocr_text, question)
    choice_score = max(seq_sim(ocr_text, c) for c in choices)

    od = extract_digits(ocr_text)
    digit_bonus = 0.0

    if len(od) >= 2:
        for choice in choices:
            cd = extract_digits(choice)
            if not cd:
                continue

            if od == cd:
                d = 1.0
            elif od in cd or cd in od:
                d = 0.80
            else:
                d = SequenceMatcher(None, od, cd).ratio()

            digit_bonus = max(digit_bonus, d)

    base = (
        0.50 * choice_score
        + 0.30 * q_score
        + 0.20 * digit_bonus
    )

    conf_factor = 0.75 + 0.25 * max(0.0, min(float(conf), 1.0))
    return base * conf_factor


def bbox_xyxy(bbox):
    xs = [float(p[0]) for p in bbox]
    ys = [float(p[1]) for p in bbox]
    return min(xs), min(ys), max(xs), max(ys)


def expanded_crop_box(bbox, w, h):
    x1,y1,x2,y2 = bbox_xyxy(bbox)
    bw = max(x2-x1, 1)
    bh = max(y2-y1, 1)

    cx = (x1+x2)/2
    cy = (y1+y2)/2

    tw = max(bw * CROP_EXPAND_X, MIN_CROP_WIDTH)
    th = max(bh * CROP_EXPAND_Y, MIN_CROP_HEIGHT)

    return (
        max(0, int(cx-tw/2)),
        max(0, int(cy-th/2)),
        min(w, int(cx+tw/2)),
        min(h, int(cy+th/2)),
    )


def get_v3_top1_crop(row, image):
    items = OCR_JSON_MAP.get(str(row["id"]), [])
    candidates = []

    for item in items:
        text = str(item.get("text",""))
        conf = float(item.get("confidence",0.0))
        bbox = item.get("bbox")

        if conf < OCR_MIN_CONF or not text.strip() or not bbox:
            continue

        candidates.append({
            "score": ocr_relevance_score(text, conf, row),
            "bbox": bbox,
        })

    if not candidates:
        return None

    candidates.sort(key=lambda x:x["score"], reverse=True)
    best = candidates[0]

    if best["score"] < MIN_RELEVANCE_SCORE:
        return None

    crop_box = expanded_crop_box(
        best["bbox"],
        image.width,
        image.height,
    )

    return image.crop(crop_box)

OCR JSON map: 16111


# 9. V3 best model 로드

In [ ]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

processor = AutoProcessor.from_pretrained(
    V3_BEST_ADAPTER,
    trust_remote_code=True,
)

processor.image_processor.size = {
    "longest_edge": MAX_PIXELS,
    "shortest_edge": MIN_PIXELS,
}

base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).to("cuda")

model = PeftModel.from_pretrained(
    base_model,
    V3_BEST_ADAPTER,
    is_trainable=False,
)

model.eval()
model.config.use_cache = True

choice_token_ids = {
    c: processor.tokenizer(c, add_special_tokens=False).input_ids
    for c in CHOICES
}

assert all(len(v) == 1 for v in choice_token_ids.values())
choice_token_ids = {k:v[0] for k,v in choice_token_ids.items()}

print("V3 loaded")
print("VRAM:", round(torch.cuda.memory_allocated()/1024**3, 2), "GB")

`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config', 'lora_ga_config', 'monteclora_config', 'use_bdlora', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


V3 loaded
VRAM: 16.51 GB


# 10. Permuted-choice 추론

In [ ]:
SYSTEM_INSTRUCT = (
    "You are a visual multiple-choice question answering assistant. "
    "Inspect all provided images carefully. "
    "The first image is the full scene. "
    "If a second image is present, it is an enlarged region that may contain "
    "text or details relevant to the question. "
    "Pay close attention to Korean/English text, numbers, phone numbers, "
    "prices, signs, menus, labels, and tables. "
    "Choose the best answer from a, b, c, or d. "
    "Output exactly one lowercase letter and nothing else."
)


def build_permuted_prompt(row, perm):
    original = [
        str(row["a"]),
        str(row["b"]),
        str(row["c"]),
        str(row["d"]),
    ]

    shown = [original[i] for i in perm]

    return (
        f"질문: {row['question']}\n\n"
        f"(a) {shown[0]}\n"
        f"(b) {shown[1]}\n"
        f"(c) {shown[2]}\n"
        f"(d) {shown[3]}\n\n"
        "첫 번째 전체 이미지와, 제공된 경우 두 번째 확대 이미지를 함께 확인하세요. "
        "정답을 a, b, c, d 중 하나의 소문자 한 글자로만 답하세요."
    )


def softmax_np(x):
    x = np.asarray(x, dtype=np.float64)
    x -= np.max(x)
    e = np.exp(x)
    return e / e.sum()


@torch.inference_mode()
def predict_permutation(row, perm):
    full = Image.open(
        resolve_image_path(row["path"])
    ).convert("RGB")

    crop = get_v3_top1_crop(row, full)

    images = [full]
    user_content = [{"type":"image","image":full}]

    if crop is not None:
        images.append(crop)
        user_content.append({"type":"image","image":crop})

    user_content.append({
        "type":"text",
        "text": build_permuted_prompt(row, perm),
    })

    messages = [
        {
            "role":"system",
            "content":[{"type":"text","text":SYSTEM_INSTRUCT}],
        },
        {
            "role":"user",
            "content":user_content,
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[text],
        images=images,
        return_tensors="pt",
    )

    inputs = {
        k: (v.to("cuda", non_blocking=True) if torch.is_tensor(v) else v)
        for k,v in inputs.items()
    }

    out = model(**inputs)

    mask = inputs["attention_mask"][0]
    last_pos = int(torch.nonzero(mask, as_tuple=False)[-1].item())
    logits = out.logits[0, last_pos]

    display_scores = np.array([
        float(logits[choice_token_ids[c]].float().item())
        for c in CHOICES
    ])

    display_probs = softmax_np(display_scores)

    # displayed position probability -> original option identity
    original_probs = np.zeros(4, dtype=np.float64)

    for display_idx, original_idx in enumerate(perm):
        original_probs[original_idx] = display_probs[display_idx]

    return original_probs

# 11. Permutation cache

In [ ]:
DEV_PERM_CACHE = os.path.join(OUTPUT_DIR, "dev_perm_cache.csv")
TEST_PERM_CACHE = os.path.join(OUTPUT_DIR, "test_perm_cache.csv")


def load_perm_cache(path):
    if os.path.isfile(path):
        x = pd.read_csv(path)
        x["id"] = x["id"].astype(str)
        return x

    return pd.DataFrame(columns=[
        "id","perm_idx","prob_a","prob_b","prob_c","prob_d"
    ])


def run_perm_cache(source_df, indices, cache_path, desc):
    cache = load_perm_cache(cache_path)

    existing = set(
        zip(
            cache["id"].astype(str),
            cache["perm_idx"].astype(int),
        )
    ) if len(cache) else set()

    new_rows = []

    total = len(indices) * len(PERMUTATIONS)

    with tqdm(total=total, desc=desc) as pbar:
        for i in indices:
            row = source_df.iloc[i]
            rid = str(row["id"])

            for perm_idx, perm in enumerate(PERMUTATIONS):
                if (rid, perm_idx) in existing:
                    pbar.update(1)
                    continue

                probs = predict_permutation(row, perm)

                new_rows.append({
                    "id":rid,
                    "perm_idx":perm_idx,
                    "prob_a":probs[0],
                    "prob_b":probs[1],
                    "prob_c":probs[2],
                    "prob_d":probs[3],
                })

                pbar.update(1)

                if len(new_rows) >= 100:
                    cache = pd.concat(
                        [cache, pd.DataFrame(new_rows)],
                        ignore_index=True,
                    )
                    cache = cache.drop_duplicates(
                        ["id","perm_idx"],
                        keep="last",
                    )
                    cache.to_csv(
                        cache_path,
                        index=False,
                        encoding="utf-8-sig",
                    )
                    existing.update(
                        (str(r["id"]), int(r["perm_idx"]))
                        for r in new_rows
                    )
                    new_rows = []

    if new_rows:
        cache = pd.concat(
            [cache, pd.DataFrame(new_rows)],
            ignore_index=True,
        )
        cache = cache.drop_duplicates(
            ["id","perm_idx"],
            keep="last",
        )
        cache.to_csv(
            cache_path,
            index=False,
            encoding="utf-8-sig",
        )

    return cache

# 12. Dev permutation 추론

In [ ]:
dev_perm_indices = np.where(
    f3_dev["margin"] <= DEV_MAX_BASE_MARGIN
)[0].tolist()

print(
    "Dev permutation 대상:",
    len(dev_perm_indices),
    "/",
    len(dev_df),
)

dev_perm = run_perm_cache(
    dev_df,
    dev_perm_indices,
    DEV_PERM_CACHE,
    "Dev choice permutation",
)

print("Cached rows:", len(dev_perm))

Dev permutation 대상: 321 / 2683


Dev choice permutation:   0%|          | 0/963 [00:00<?, ?it/s]

/tmp/ipykernel_1289/4005420883.py:54: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cache = pd.concat(


Cached rows: 963


# 13. Dev 4-view 결합 / search

In [ ]:
def build_perm_mean_map(cache):
    out = {}

    for rid, g in cache.groupby("id"):
        if len(g) < 3:
            continue

        g = g.sort_values("perm_idx")

        p = g[
            ["prob_a","prob_b","prob_c","prob_d"]
        ].to_numpy(dtype=float)

        out[str(rid)] = {
            "mean": p.mean(axis=0),
            "views": p,
            "preds": p.argmax(axis=1),
        }

    return out


dev_perm_map = build_perm_mean_map(dev_perm)

print("Complete Dev items:", len(dev_perm_map))


def make_blend_preds(base_probs, source_df, perm_map, margin_th, base_weight):
    pred = base_probs.argmax(axis=1).copy()

    for i in range(len(source_df)):
        if f3_dev["margin"][i] > margin_th:
            continue

        rid = str(source_df.iloc[i]["id"])

        if rid not in perm_map:
            continue

        perm_mean = perm_map[rid]["mean"]

        final_p = (
            base_weight * base_probs[i]
            + (1.0 - base_weight) * perm_mean
        )

        pred[i] = int(np.argmax(final_p))

    return np.array([CHOICES[i] for i in pred])


search_rows = []

for margin_th in MARGIN_THRESHOLDS:
    for base_w in BASE_WEIGHTS:
        pred = make_blend_preds(
            p3_dev,
            dev_df,
            dev_perm_map,
            margin_th,
            base_w,
        )

        metrics = evaluate_dev(pred)

        search_rows.append({
            "margin_th":margin_th,
            "base_weight":base_w,
            "changed_rate":float(np.mean(pred != f3_dev["pred"])),
            "changed_n":int(np.sum(pred != f3_dev["pred"])),
            **metrics,
        })


search_df = pd.DataFrame(search_rows)
display(
    search_df.sort_values(
        ["consensus","high_conf","majority"],
        ascending=False,
    ).head(30)
)

Complete Dev items: 321


,margin_th,base_weight,changed_rate,changed_n,consensus,majority,high_conf
35,0.50,0.25,0.048081,129,0.477558,0.689982,0.704612
37,0.50,0.45,0.037644,101,0.477558,0.691336,0.706575
32,0.40,0.45,0.036154,97,0.477409,0.690884,0.706084
30,0.40,0.25,0.043235,116,0.477390,0.689079,0.704122
38,0.50,0.55,0.032799,88,0.477372,0.690884,0.706084
36,0.50,0.35,0.044726,120,0.477316,0.689531,0.704612
31,0.40,0.35,0.041372,111,0.477316,0.689079,0.704122
33,0.40,0.55,0.032426,87,0.477148,0.690433,0.705594
15,0.20,0.25,0.027581,74,0.477017,0.689079,0.704122
18,0.20,0.55,0.023481,63,0.476905,0.689982,0.705103


# 14. Conservative / Balanced / Aggressive 설정 자동 선택

Dev 최고값 하나만 고르면 과적합 위험이 있으므로,
V3 대비 변경률에 cap을 둔 3개 후보를 각각 선택합니다.

- conservative: 최대 3%
- balanced: 최대 8%
- aggressive: 최대 15%

In [ ]:
def choose_under_change_cap(cap):
    x = search_df[
        search_df["changed_rate"] <= cap
    ].copy()

    if len(x) == 0:
        x = search_df.copy()

    x = x.sort_values(
        ["consensus","high_conf","majority"],
        ascending=False,
    )

    return x.iloc[0]


best_cons = choose_under_change_cap(0.03)
best_bal = choose_under_change_cap(0.08)
best_aggr = choose_under_change_cap(0.15)

selected = pd.DataFrame([
    {"candidate":"conservative", **best_cons.to_dict()},
    {"candidate":"balanced", **best_bal.to_dict()},
    {"candidate":"aggressive", **best_aggr.to_dict()},
])

display(selected)

,candidate,margin_th,base_weight,changed_rate,changed_n,consensus,majority,high_conf
0,conservative,0.2,0.25,0.027581,74.0,0.477017,0.689079,0.704122
1,balanced,0.5,0.25,0.048081,129.0,0.477558,0.689982,0.704612
2,aggressive,0.5,0.25,0.048081,129.0,0.477558,0.689982,0.704612


# 15. Test에서 필요한 최대 margin까지만 추가 추론

In [ ]:
selected_margin_max = max(
    float(best_cons["margin_th"]),
    float(best_bal["margin_th"]),
    float(best_aggr["margin_th"]),
)

test_perm_indices = np.where(
    f3_test["margin"] <= selected_margin_max
)[0].tolist()

print("선택된 최대 margin:", selected_margin_max)
print(
    "Test permutation 대상:",
    len(test_perm_indices),
    "/",
    len(test_df),
)
print(
    "추가 forward passes:",
    len(test_perm_indices) * 3,
)

선택된 최대 margin: 0.5
Test permutation 대상: 130 / 6714
추가 forward passes: 390


# 16. Test permutation 추론

In [ ]:
test_perm = run_perm_cache(
    test_df,
    test_perm_indices,
    TEST_PERM_CACHE,
    "Test choice permutation",
)

test_perm_map = build_perm_mean_map(test_perm)

print("Complete Test items:", len(test_perm_map))

Test choice permutation:   0%|          | 0/390 [00:00<?, ?it/s]

/tmp/ipykernel_1289/4005420883.py:54: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cache = pd.concat(


Complete Test items: 130


# 17. Submission 후보 3개 생성

In [ ]:
def make_test_candidate(row):
    margin_th = float(row["margin_th"])
    base_w = float(row["base_weight"])

    pred_idx = p3_test.argmax(axis=1).copy()

    for i in range(len(test_df)):
        if f3_test["margin"][i] > margin_th:
            continue

        rid = str(test_df.iloc[i]["id"])

        if rid not in test_perm_map:
            continue

        final_p = (
            base_w * p3_test[i]
            + (1.0-base_w) * test_perm_map[rid]["mean"]
        )

        pred_idx[i] = int(np.argmax(final_p))

    return np.array([CHOICES[i] for i in pred_idx])


pred_cons = make_test_candidate(best_cons)
pred_bal = make_test_candidate(best_bal)
pred_aggr = make_test_candidate(best_aggr)

candidate_00 = pd.DataFrame({
    "id":test_df["id"],
    "answer":f3_test["pred"],
})

candidate_01 = pd.DataFrame({
    "id":test_df["id"],
    "answer":pred_cons,
})

candidate_02 = pd.DataFrame({
    "id":test_df["id"],
    "answer":pred_bal,
})

candidate_03 = pd.DataFrame({
    "id":test_df["id"],
    "answer":pred_aggr,
})

paths = {
    "00_v3_reproduction.csv":candidate_00,
    "01_perm_conservative.csv":candidate_01,
    "02_perm_balanced.csv":candidate_02,
    "03_perm_aggressive.csv":candidate_03,
}

for name, df in paths.items():
    p = os.path.join(OUTPUT_DIR, name)
    df.to_csv(p, index=False, encoding="utf-8-sig")
    print("Saved:", p)

Saved: /content/drive/MyDrive/Qwen3VL_FINAL2_PERM_TTA/00_v3_reproduction.csv
Saved: /content/drive/MyDrive/Qwen3VL_FINAL2_PERM_TTA/01_perm_conservative.csv
Saved: /content/drive/MyDrive/Qwen3VL_FINAL2_PERM_TTA/02_perm_balanced.csv
Saved: /content/drive/MyDrive/Qwen3VL_FINAL2_PERM_TTA/03_perm_aggressive.csv


# 18. Vote-switch 후보

In [ ]:
# 3개 permutation view 중 2개 이상이 같은 alternative를 지지하고
# base가 low-margin일 때만 매우 보수적으로 switch.

vote_pred = f3_test["pred"].copy()

for i in test_perm_indices:
    rid = str(test_df.iloc[i]["id"])

    if rid not in test_perm_map:
        continue

    info = test_perm_map[rid]
    mapped_preds = info["preds"]

    counts = Counter(mapped_preds.tolist())
    alt_idx, n_votes = counts.most_common(1)[0]

    base_idx = int(np.argmax(p3_test[i]))

    if (
        n_votes >= 2
        and alt_idx != base_idx
        and f3_test["margin"][i] <= selected_margin_max
    ):
        # 평균 permutation probability도 base보다 충분히 강한 경우
        perm_mean = info["mean"]

        if (
            perm_mean[alt_idx]
            >= perm_mean[base_idx] + 0.08
        ):
            vote_pred[i] = CHOICES[alt_idx]


candidate_04 = pd.DataFrame({
    "id":test_df["id"],
    "answer":vote_pred,
})

path04 = os.path.join(
    OUTPUT_DIR,
    "04_perm_vote_switch.csv",
)

candidate_04.to_csv(
    path04,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", path04)

Saved: /content/drive/MyDrive/Qwen3VL_FINAL2_PERM_TTA/04_perm_vote_switch.csv


# 19. 후보 비교

In [ ]:
candidate_preds = {
    "00_V3":f3_test["pred"],
    "01_conservative":pred_cons,
    "02_balanced":pred_bal,
    "03_aggressive":pred_aggr,
    "04_vote_switch":vote_pred,
}

rows = []

for name, pred in candidate_preds.items():
    rows.append({
        "candidate":name,
        "changed_n":int(np.sum(pred != f3_test["pred"])),
        "changed_rate":float(np.mean(pred != f3_test["pred"])),
        "a":int(np.sum(pred=="a")),
        "b":int(np.sum(pred=="b")),
        "c":int(np.sum(pred=="c")),
        "d":int(np.sum(pred=="d")),
    })

summary = pd.DataFrame(rows)
display(summary)

summary.to_csv(
    os.path.join(OUTPUT_DIR, "candidate_summary.csv"),
    index=False,
    encoding="utf-8-sig",
)

selected.to_csv(
    os.path.join(OUTPUT_DIR, "dev_selected_params.csv"),
    index=False,
    encoding="utf-8-sig",
)

,candidate,changed_n,changed_rate,a,b,c,d
0,00_V3,0,0.000000,1734,1664,1624,1692
1,01_conservative,23,0.003426,1726,1662,1626,1700
2,02_balanced,42,0.006256,1718,1667,1627,1702
3,03_aggressive,42,0.006256,1718,1667,1627,1702
4,04_vote_switch,35,0.005213,1725,1669,1624,1696


# 20. 제출 순서

기존 최고 V3 `0.95918`은 반드시 남겨둡니다.

### 제출 1회 가능
`01_perm_conservative.csv`

### 2회 가능
1. `01_perm_conservative.csv`
2. `02_perm_balanced.csv`

### 3회 가능
3. `04_perm_vote_switch.csv`

`03_perm_aggressive.csv`는 제출 횟수가 충분할 때만 권장합니다.

---

## 판단 기준

Candidate summary에서:

```text
changed_rate < 3%   : 매우 보수적
3~8%                : 적당
8~15%               : 공격적
```

현재 Public score가 이미 높기 때문에,
많은 문제를 뒤집는 것보다 **틀릴 가능성이 높은 low-margin 문제만 고치는 쪽**에 집중했습니다.